## Dependências

In [17]:
import time as ti
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.edge.service import Service
from selenium.webdriver.support.ui import Select

## Funções

In [18]:
#TODO Criar função para scrapping;
#TODO Criar uma função para ir para próxima página;

## Variáveis

In [143]:
URL_BASE = "https://www3.tjrj.jus.br/ejuris/ConsultarJurisprudencia.aspx"
driver_path = 'msedgedriver.exe'
service = Service(driver_path)
driver = webdriver.Edge(service=service)

## Funções

In [20]:
def search(term: str):
    search_box_name = "ctl00$ContentPlaceHolder1$txtTextoPesq"

    search_box = driver.find_element(By.NAME, search_box_name)
    search_box.send_keys(term)
    search_box.send_keys(Keys.ENTER)

In [140]:
def get_result_tables() -> list | None:
    tables_xpath = "/html/body/div[1]/div/div[2]/form/div[3]/span/table/tbody/tr[6]/td"
    
    elem = driver.find_element(By.XPATH, tables_xpath)
    html_content = elem.get_attribute("innerHTML")
    
    if html_content:
        soup = BeautifulSoup(html_content, "html.parser")
        tables = soup.find_all("table")

        data = []

        for table in tables:
            lines = table.find_all("tr")
            
            # Ignora tabelas menores da estrutura do site (ruídos)
            if len(lines) < 16:
                continue
            

            # Se algum dia precisar usar todas as datas (caso venha mais de uma)
            # linhas_decisao = []
            # for line in lines[10]:
            #     texto = line.get_text(strip=True)
            #     if "Data de Julgamento:" in texto:
            #         linhas_decisao.append(texto)
                    
            
            processo_info = lines[3].get_text(strip=True)
            processo_num, processo_tipo = processo_info.split("- ")
            
            ementa = lines[5].get_text(strip=True)
            
            julgamento_info = lines[6].get_text(strip=True)
            desembargador, _, camara = julgamento_info.split(" - ")
            desembargador = desembargador.removeprefix("Des(a). ")
            
            resumo = lines[10].get_text(" ", strip=True)  # usa separador para manter espaços
            
            if len(lines) > 16:
                detalhes_decisao = lines[16].get_text(strip=True)
            else:
                detalhes_decisao = lines[13].get_text(strip=True)
                
            detalhes_decisao = detalhes_decisao.removesuffix("(*)").split("-")
            
            decisao = detalhes_decisao[0]
            julg_data = detalhes_decisao[1].strip().replace("Data de Julgamento:", "").strip()
            publicacao_data = detalhes_decisao[2].replace("Data de Publicação:", "").strip()
            
            # Adiciona no dataset
            data.append({
                "Número do Processo": processo_num,
                "Tipo do Processo": processo_tipo,
                "Ementa": ementa,
                "Desembargador": desembargador,
                "Câmara": camara,
                "Resumo": resumo,
                "Data do Julgamento": julg_data,
                "Data da Publicação": publicacao_data,
                "Decisão": decisao
            })
            
            #mapping debug
            # mapping = []
            # for idx, line in enumerate(lines):
            #     if not line.get_text(strip=True):
            #         continue
            #     line_dict = {"name": line.get_text(strip=True), "line":idx}

            #     mapping.append(line_dict)
                
            # display(pd.DataFrame(mapping))
        
        return data

## Inicio

In [144]:
driver.get(URL_BASE)

search("danos morais")

#df = pd.DataFrame(data)

#display(df)

In [145]:
result = get_result_tables()

df_result = pd.DataFrame(result)
df_result

,Número do Processo,Tipo do Processo,Ementa,Desembargador,Câmara,Resumo,Data do Julgamento,Data da Publicação,Decisão
0,0009137-93.2026.8.19.0000,AGRAVO DE INSTRUMENTO,1ª Ementa,EDUARDO DE AZEVEDO PAIVA,TERCEIRA CAMARA DE DIREITO PRIVADO (ANTIGA 18ª...,AGRAVO DE INSTRUMENTO. AÇÃO DECLARATÓRIA DE IN...,03/03/2026,05/03/2026,Decisão monocrática
1,0029580-56.2017.8.19.0202,APELAÇÃO,1ª Ementa,EDUARDO DE AZEVEDO PAIVA,TERCEIRA CAMARA DE DIREITO PRIVADO (ANTIGA 18ª...,APELAÇÃO CÍVEL. DIREITO DO CONSUMIDOR. AÇÃO DE...,03/03/2026,05/03/2026,Decisão monocrática
2,0053678-95.2019.8.19.0021,APELAÇÃO,1ª Ementa,SIRLEY ABREU BIONDI,SEXTA CAMARA DE DIREITO PRIVADO (ANTIGA 13ª CÂ...,Ação declaratória de inexistência de débito c/...,03/03/2026,05/03/2026,Decisão monocrática
3,0804792-16.2025.8.19.0023,APELAÇÃO,1ª Ementa,MARCIA FERREIRA ALVARENGA,OITAVA CAMARA DE DIREITO PRIVADO (ANTIGA 17ª C...,APELAÇÃO CÍVEL. CÂMARA DE DIREITO PRIVADO. RE...,03/03/2026,05/03/2026,Íntegra do \t\t\t\tAcórdão
4,0803435-80.2024.8.19.0202,APELAÇÃO,1ª Ementa,MARCIA FERREIRA ALVARENGA,OITAVA CAMARA DE DIREITO PRIVADO (ANTIGA 17ª C...,APELAÇÃO CÍVEL. CÂMARA DE DIREITO PRIVADO. DIR...,03/03/2026,05/03/2026,Íntegra do \t\t\t\tAcórdão
5,0801984-53.2023.8.19.0073,APELAÇÃO,1ª Ementa,MARCIA FERREIRA ALVARENGA,OITAVA CAMARA DE DIREITO PRIVADO (ANTIGA 17ª C...,APELAÇÃO CÍVEL. CÂMARA DE DIREITO PRIVADO. DIR...,03/03/2026,05/03/2026,Íntegra do \t\t\t\tAcórdão
6,0024600-22.2020.8.19.0021,APELAÇÃO,1ª Ementa,MARCIA FERREIRA ALVARENGA,OITAVA CAMARA DE DIREITO PRIVADO (ANTIGA 17ª C...,APELAÇÃO CÍVEL. CÂMARA DE DIREITO PRIVADO. DIR...,03/03/2026,05/03/2026,Íntegra do \t\t\t\tAcórdão
7,0863320-32.2024.8.19.0038,APELAÇÃO,1ª Ementa,MARCIA FERREIRA ALVARENGA,OITAVA CAMARA DE DIREITO PRIVADO (ANTIGA 17ª C...,APELAÇÃO CÍVEL. CÂMARA DE DIREITO PRIVADO. AÇÃ...,03/03/2026,05/03/2026,Íntegra do \t\t\t\tAcórdão
8,0828022-18.2023.8.19.0004,APELAÇÃO,1ª Ementa,MARCIA FERREIRA ALVARENGA,OITAVA CAMARA DE DIREITO PRIVADO (ANTIGA 17ª C...,APELAÇÃO CÍVEL. CÂMARA DE DIREITO PRIVADO. DIR...,03/03/2026,05/03/2026,Íntegra do \t\t\t\tAcórdão
9,0800914-50.2024.8.19.0013,APELAÇÃO,1ª Ementa,LUIZ FERNANDO DE ANDRADE PINTO,TERCEIRA CAMARA DE DIREITO PRIVADO (ANTIGA 18ª...,APELAÇÃO CÍVEL. AÇÃO DE OBRIGAÇÃO DE FAZER C/C...,03/03/2026,05/03/2026,Decisão monocrática


In [ ]:
page_xpath = '//*[@id="seletorPaginasRodape"]'
select_element = driver.find_element(By.XPATH, page_xpath)

select = Select(select_element)

select.select_by_value("3")

In [ ]:
#driver.quit()